# 4. RQ5: Capacidades sensíveis à segurança e sua deriva nas cópias

**Pergunta:** as skills mandam o agente fazer coisas perigosas? E, quando uma skill é copiada e editada, a cópia fica mais perigosa?

**Por que importa:** o agente executa o que a skill manda, com as permissões do usuário. Uma cópia editada que acrescenta `curl … | sh` ou `--dangerously-skip-permissions` é o equivalente, sem registro e sem revisão, a um pacote adulterado na cadeia de suprimentos.

**Como medimos:** 9 categorias de capacidade detectadas por expressões regulares no texto, cada uma com nível de gravidade e a CWE mais próxima. (1) prevalência; (2) deriva: o que cada variante acrescenta ou remove em relação à versão mais copiada da família, com teste de McNemar; (3) regressão logística para ver que tipo de variante acrescenta capacidades.

**Importante:** medimos **capacidade** (potencial de dano), não vulnerabilidade nem ataque confirmado.

---

### English summary

**What we measure.** Not vulnerabilities and not attacks, but *security-sensitive capabilities*: instructions that give the agent the power to cause harm if the skill were malicious, tampered with in a copy, or misused. Each capability maps to the closest CWE weakness class and a severity tier.

**Methodology step 3: quantitative + statistical modelling.**

* RQ5.1 Prevalence of each capability, also weighted by copies (exposure).
* RQ5.2 Drift: within each variant family, which capabilities an edited copy **adds** or **removes** relative to the most-copied member. Additions vs removals are tested with an exact McNemar test.
* RQ5.3 Logistic regression: which variants add capabilities?

**Methodology step 4 (validation)** happens in notebook 05: manual labelling to measure detector precision.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as smf

database: /Users/cleber/Documents/gitskills-mc/code/data/raw/agent_skills_release.db


In [2]:
skills = clean.strict(clean.load_skills())
risk_cols = [c for c in skills.columns if c.startswith("risk_")]
pd.DataFrame({"capability": [c[5:] for c in risk_cols],
              "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
              "closest CWE": [patterns.RISK_CWE[c[5:]] for c in risk_cols]})

,capability,severity,closest CWE
0,pipe_to_shell,high,CWE-494 Download of Code Without Integrity Check
1,remote_fetch,low,CWE-829 Inclusion of Functionality from Untrusted Control Sphere
2,destructive_fs,high,CWE-732 Incorrect Permission Assignment for Critical Resource
3,privilege,medium,CWE-250 Execution with Unnecessary Privileges
4,dynamic_exec,high,CWE-95 Eval Injection / CWE-78 OS Command Injection
5,package_install,medium,CWE-1357 Reliance on Insufficiently Trustworthy Component
6,secrets,medium,CWE-522 Insufficiently Protected Credentials
7,git_push,low,CWE-284 Improper Access Control
8,safety_bypass,high,CWE-862 Missing Authorization


A tabela acima é o "dicionário" da RQ5: cada capacidade, sua gravidade (alta, média, baixa) e a classe de fraqueza CWE mais próxima. Os padrões exatos estão em `src/gitskills/patterns.py`.

## 4.1 Prevalência e exposição
**Objetivo:** saber quão comum é cada capacidade. `share_of_skills` conta cada texto uma vez; `share_of_occurrences` pondera pelo número de cópias, ou seja, quantas instalações carregam a capacidade (exposição).

In [3]:
prev = pd.DataFrame({
    "severity": [patterns.RISK_SEVERITY[c[5:]] for c in risk_cols],
    "share_of_skills": skills[risk_cols].mean().values,
    "share_of_occurrences": (skills[risk_cols].mul(skills["copies"], axis=0).sum() / skills["copies"].sum()).values,
}, index=[c[5:] for c in risk_cols]).sort_values(["severity", "share_of_skills"], ascending=[True, False])
save_table(prev.reset_index(names="capability"), "rq5_risk_prevalence")
high = [c for c in risk_cols if patterns.RISK_SEVERITY[c[5:]] == "high"]
print(f"skills with >=1 high-severity capability: {skills[high].any(axis=1).mean():.1%}")
prev

  table  -> results/tables/rq5_risk_prevalence.csv
skills with >=1 high-severity capability: 6.4%


,severity,share_of_skills,share_of_occurrences
dynamic_exec,high,0.044011,0.044458
destructive_fs,high,0.011072,0.011738
safety_bypass,high,0.006719,0.005770
pipe_to_shell,high,0.006703,0.007479
remote_fetch,low,0.085654,0.096361
git_push,low,0.037503,0.033981
secrets,medium,0.176242,0.199352
package_install,medium,0.119460,0.149851
privilege,medium,0.020894,0.025083


**O que os dados mostram:**
* **6,4%** das skills têm pelo menos uma capacidade de **gravidade alta**. A mais comum é a execução dinâmica (4,4%); `curl | sh` e desligar confirmações aparecem em 0,7% cada.
* As mais frequentes no geral são de gravidade média: mencionar segredos (**17,6%**) e instalar pacotes (**11,9%**).
* A exposição (por cópias) é um pouco **maior** que a prevalência em várias categorias (instalar pacotes: 11,9% → 15,0%): skills com essas capacidades tendem a ser mais copiadas.

**Cuidado: estes números são detecções, não instruções confirmadas.** A validação do notebook 05 (12 trechos por categoria, rotulados por dois avaliadores e decididos por consenso; κ = 0,75 na decisão instrução ou não) mostra precisões muito diferentes entre categorias:

| Capacidade | Detectado | Precisão | Prevalência ajustada |
|---|---|---|---|
| `curl \| sh` | 0,7% | 1,00 | **0,7%** |
| instalar pacotes | 11,9% | 0,83 | **~10%** |
| apagar arquivos | 1,1% | 0,83 | ~0,9% |
| privilégio | 2,1% | 0,75 | ~1,6% |
| segredos | 17,6% | 0,17 | ~3% |
| execução dinâmica | 4,4% | 0,08 | ~0,4% |
| desligar confirmações | 0,7% | 0,33 | ~0,2% |

Os detectores de `curl | sh`, instalação de pacotes, apagar arquivos e privilégio são confiáveis. Os de segredos, execução dinâmica e desligar confirmações disparam sobretudo em **menções** ("never store credentials", "never use `--no-verify`") ou em outro sentido da palavra ("eval" de *evaluation*). Com a precisão descontada, a fração de skills com alguma capacidade de gravidade alta cai de 6,4% para **no máximo ~2%**.

## 4.2 Deriva nas cópias editadas
**Objetivo:** a pergunta central. Em cada família, a referência é a variante mais copiada (proxy da original, que a base não identifica). Para cada outra variante, marcamos que capacidades ela **acrescentou** e quais **removeu**.

**Teste de McNemar:** se as edições fossem aleatórias em relação ao risco, acrescentar e remover seriam igualmente prováveis (50/50). Um p pequeno indica que um dos lados predomina.

In [4]:
fam = reuse.name_families(skills)
drift = reuse.variant_drift(fam, risk_cols)
k, n = int(drift["any_added"].sum()), len(drift)
lo, hi = proportion_confint(k, n, method="wilson")
print(f"variants adding >=1 capability: {k:,}/{n:,} = {k/n:.1%} (95% CI {lo:.1%}-{hi:.1%})")
rows = []
for c in risk_cols:
    a, r = int(drift[f"added_{c}"].sum()), int(drift[f"removed_{c}"].sum())
    rows.append({"capability": c[5:], "severity": patterns.RISK_SEVERITY[c[5:]], "added": a, "removed": r,
                 "mcnemar_p": binomtest(a, a + r, 0.5).pvalue if a + r else np.nan})
mc = pd.DataFrame(rows)
save_table(mc, "rq5_mcnemar_added_vs_removed")
mc

variants adding >=1 capability: 136,096/788,712 = 17.3% (95% CI 17.2%-17.3%)
  table  -> results/tables/rq5_mcnemar_added_vs_removed.csv


,capability,severity,added,removed,mcnemar_p
0,pipe_to_shell,high,3429,3175,1.848117e-03
1,remote_fetch,low,31683,32211,3.707958e-02
2,destructive_fs,high,5055,5741,4.258280e-11
3,privilege,medium,8064,8347,2.771091e-02
4,dynamic_exec,high,19016,15605,3.857394e-75
5,package_install,medium,40320,51713,2.169481e-309
6,secrets,medium,57422,65252,9.589902e-111
7,git_push,low,17793,17840,8.074740e-01
8,safety_bypass,high,4477,5527,8.873303e-26


**O que os dados mostram:**
* **17,3%** das 788.712 variantes (IC 95%: 17,2–17,3%) acrescentam pelo menos uma capacidade que a referência não tinha.
* Mas a deriva vai **nos dois sentidos**. Na maioria das categorias as edições **mais removem do que acrescentam** (instalar pacotes: 40.320 acréscimos × 51.713 remoções; segredos: 57.422 × 65.252; desligar confirmações: 4.477 × 5.527).
* **A exceção confiável é `curl | sh`:** baixar um script e executá-lo direto é acrescentado mais vezes do que removido (3.429 × 3.175, p = 0,002), e o detector teve precisão de 12/12 na validação.
* A execução dinâmica também aparece como "mais acrescentada" (19.016 × 15.605), mas **não deve ser interpretada**: o detector acerta só 1 em 12, porque a palavra "eval" quase sempre significa *evaluation* (avaliação de modelos). O efeito provavelmente reflete skills que passaram a falar de avaliação.
* Nas categorias de baixa precisão (segredos, desligar confirmações), parte das "remoções" pode ser só uma **regra** do tipo "nunca use `--no-verify`" que sumiu do texto.

Leitura: copiar e editar **não** torna as skills mais perigosas de forma geral; as cópias tendem a ser enxugadas. A forma mais direta de executar código remoto, `curl | sh`, é a que se acumula.

## 4.2b Saldo por variante: as cópias ficam mais ou menos capazes?
**Objetivo:** o McNemar olha uma capacidade de cada vez. Aqui juntamos todas num **saldo por variante**: (nº de capacidades da cópia) − (nº da referência). Com isso, cada variante cai num de quatro grupos: só acrescentou, só removeu, as duas coisas, nada mudou.

**Teste de Wilcoxon (signed-rank):** verifica se os saldos diferentes de zero tendem a ser positivos (as cópias ganham capacidades) ou negativos (perdem). O tamanho do efeito é a correlação rank-biserial *r*, que vai de −1 (todas perdem) a +1 (todas ganham). Rodamos duas versões: com as 9 capacidades e **só com as 4 de detector confiável** (precisão ≥ 0,75 no notebook 05), para que a conclusão não dependa dos detectores ruins.

In [5]:
from scipy.stats import wilcoxon
rows, groups = [], {}
for label, caps in [("all 9 capabilities", [c[5:] for c in risk_cols]), ("4 reliable detectors", list(reuse.RELIABLE_CAPABILITIES))]:
    b = reuse.drift_balance(drift, caps)
    groups[label] = b["group"].value_counts(normalize=True)
    nz = b.loc[b["net"] != 0, "net"]
    stat = wilcoxon(nz, zero_method="wilcox")
    ranks = nz.abs().rank()
    r_rb = (ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum()
    rows.append({"capabilities": label, "variants": len(b), "net_nonzero": len(nz),
                 "net_positive": int((nz > 0).sum()), "net_negative": int((nz < 0).sum()),
                 "mean_net": b["net"].mean(), "wilcoxon_p": stat.pvalue, "rank_biserial_r": r_rb})
bal = pd.DataFrame(rows)
save_table(bal, "rq5_drift_balance_wilcoxon")
grp = pd.DataFrame(groups).rename_axis("group").reset_index()
save_table(grp, "rq5_drift_groups")
display(grp.round(3))
bal.round(4)

  table  -> results/tables/rq5_drift_balance_wilcoxon.csv
  table  -> results/tables/rq5_drift_groups.csv


,group,all 9 capabilities,4 reliable detectors
0,unchanged,0.684,0.861
1,removed only,0.143,0.075
2,added only,0.136,0.062
3,added and removed,0.036,0.003


,capabilities,variants,net_nonzero,net_positive,net_negative,mean_net,wilcoxon_p,rank_biserial_r
0,all 9 capabilities,788712,231109,112652,118457,-0.0230,0.0,-0.0491
1,4 reliable detectors,788712,107915,48737,59178,-0.0154,0.0,-0.1012


**O que os dados mostram:**
* Com as 9 capacidades: **68,5%** das variantes não mudam nenhuma capacidade, **14,3%** só removem, **13,6%** só acrescentam e **3,6%** fazem as duas coisas.
* Entre as variantes cujo saldo muda, **118.457 perdem** e **112.652 ganham** capacidades (Wilcoxon p < 0,001), mas o efeito é **muito pequeno** (r = −0,05).
* Só com os 4 detectores confiáveis: **86,1%** não mudam, 7,5% só removem, 6,2% só acrescentam; r = −0,10, também pequeno e na mesma direção.

Leitura: copiar e editar **quase nunca muda** as capacidades de uma skill, e quando muda tende **levemente** a remover. Não há piora geral, mas também não há uma "limpeza" forte. O risco está em capacidades específicas (`curl | sh`, §4.2), não no saldo.

## 4.2c Risco por popularidade
**Objetivo:** responder a uma objeção natural: uma skill muito copiada é vista por mais gente e, por isso, talvez seja **mais segura**. Se for assim, revisar só as populares não basta. Dividimos as skills por faixa de cópias e medimos (1) a taxa de cada capacidade confiável e (2) que fatia das instalações com `curl | sh` cada faixa carrega.

In [6]:
tiers = pd.cut(skills["copies"], [0, 1, 9, 99, np.inf], labels=["1", "2-9", "10-99", "100+"])
rel = [f"risk_{c}" for c in reuse.RELIABLE_CAPABILITIES]
pop = skills.groupby(tiers, observed=True).agg(skills=("file_sha", "size"), occurrences=("copies", "sum"),
                                               **{c[5:]: (c, "mean") for c in rel})
pop["share_of_occurrences"] = pop["occurrences"] / pop["occurrences"].sum()
p2s = (skills["copies"] * skills["risk_pipe_to_shell"].fillna(False)).groupby(tiers, observed=True).sum()
pop["share_of_pipe_to_shell_exposure"] = p2s / p2s.sum()
save_table(pop.rename_axis("copies_tier").reset_index(), "rq5_risk_by_popularity")
pop.round(4)

  table  -> results/tables/rq5_risk_by_popularity.csv


,skills,occurrences,pipe_to_shell,package_install,destructive_fs,privilege,share_of_occurrences,share_of_pipe_to_shell_exposure
copies,,,,,,,,
1,1282448,1282448,0.0063,0.1156,0.0109,0.0206,0.3781,0.3180
2-9,294336,956599,0.0082,0.1283,0.0116,0.0211,0.2821,0.3312
10-99,31943,837531,0.0101,0.1859,0.0133,0.0305,0.2470,0.3095
100+,2112,314796,0.0033,0.2069,0.0133,0.0379,0.0928,0.0412


**O que os dados mostram:**
* As skills mais copiadas (100+ cópias) têm a **menor** taxa de `curl | sh` (0,33%, contra 0,63% nas nunca copiadas e **1,01%** nas com 10–99 cópias). Isso confirma a objeção: popularidade parece filtrar o pior.
* Mas elas carregam só **4%** das instalações com `curl | sh`. Os outros 96% estão espalhados pelas skills pouco ou medianamente copiadas.
* Instalar pacotes e pedir privilégio **crescem** com a popularidade (instalar pacotes: 11,6% → 20,7%): as skills famosas são ferramentas completas (`pdf`, `xlsx`) que instalam dependências.

Leitura: revisar só as populares limita o alcance de uma falha numa skill famosa, mas deixa quase toda a exposição a `curl | sh` descoberta. A proteção precisa de verificação automática na cauda.

## 4.3 Que variantes acrescentam capacidades? (regressão logística)
**Objetivo:** ir além da média e ver **que características** se associam a uma variante acrescentar capacidade.

**Resposta (0/1):** a variante acrescenta pelo menos uma capacidade. **Preditores:** popularidade da referência (log de cópias), tamanho da família, se a variante é copiada entre donos, se traz scripts, se foi escrita com assistente.

**Como ler a razão de chances (odds ratio, OR):** OR = 1 não muda nada; OR = 1,46 aumenta a chance em 46%; OR = 0,69 reduz em 31%.

In [7]:
d = drift.join(skills.set_index("file_sha")[["n_owners", "has_scripts", "first_commit_ai"]], on="file_sha")
d["family_size"] = d.groupby("family")["file_sha"].transform("size") + 1
d = d.assign(added=d["any_added"].astype(int), log_ref_copies=np.log1p(d["ref_copies"]),
             log_family_size=np.log(d["family_size"]), cross_owner=(d["n_owners"] > 1).astype(int),
             has_scripts=d["has_scripts"].fillna(0).astype(int), ai_authored=d["first_commit_ai"].notna().astype(int))
model = smf.logit("added ~ log_ref_copies + log_family_size + cross_owner + has_scripts + ai_authored", data=d).fit(disp=False)
odds = pd.DataFrame({"odds_ratio": np.exp(model.params), "ci_low": np.exp(model.conf_int()[0]),
                     "ci_high": np.exp(model.conf_int()[1]), "p": model.pvalues})
save_table(odds.reset_index(names="term"), "rq5_logit_odds_ratios")
odds.round(3)

  table  -> results/tables/rq5_logit_odds_ratios.csv


,odds_ratio,ci_low,ci_high,p
Intercept,0.182,0.180,0.184,0.000
log_ref_copies,0.691,0.687,0.696,0.000
log_family_size,1.364,1.357,1.370,0.000
cross_owner,0.968,0.950,0.987,0.001
has_scripts,1.465,1.440,1.490,0.000
ai_authored,1.218,1.192,1.245,0.000


In [8]:
d["family_code"] = d["family"].astype("category").cat.codes
robust = smf.logit("added ~ log_ref_copies + log_family_size + cross_owner + has_scripts + ai_authored", data=d).fit(
    disp=False, cov_type="cluster", cov_kwds={"groups": d["family_code"]})
ci = np.exp(robust.conf_int())
odds_cl = pd.DataFrame({"odds_ratio": np.exp(robust.params), "ci_low_cluster": ci[0], "ci_high_cluster": ci[1], "p_cluster": robust.pvalues})
save_table(odds_cl.reset_index(names="term"), "rq5_logit_odds_ratios_cluster")
print(f"McFadden pseudo-R2: {model.prsquared:.3f}")
odds_cl.round(3)

  table  -> results/tables/rq5_logit_odds_ratios_cluster.csv


McFadden pseudo-R2: 0.028


,odds_ratio,ci_low_cluster,ci_high_cluster,p_cluster
Intercept,0.182,0.169,0.196,0.000
log_ref_copies,0.691,0.666,0.718,0.000
log_family_size,1.364,1.317,1.412,0.000
cross_owner,0.968,0.932,1.006,0.101
has_scripts,1.465,1.379,1.555,0.000
ai_authored,1.218,1.175,1.263,0.000


**Teste de robustez:** variantes da mesma família não são independentes, o que deixa os intervalos acima estreitos demais. Acima, os mesmos coeficientes com erros-padrão **agrupados por família** (*cluster-robust*): as razões de chances não mudam, só os intervalos.

**O que os dados mostram:** com erros agrupados por família, os intervalos alargam cerca de 2 a 3 vezes, mas todas as conclusões se mantêm, **exceto** cópia entre donos: o IC passa a cruzar 1 (0,93–1,01), ou seja, **sem efeito**.

**O que os dados mostram:**
* **Ter scripts** aumenta a chance de acrescentar capacidade em **46%** (OR 1,46).
* **Ser escrita com assistente** aumenta em **22%** (OR 1,22).
* **Famílias maiores** acrescentam mais (OR 1,36 por unidade de log).
* Variantes de skills **muito populares** acrescentam **menos** (OR 0,69): quem copia uma skill famosa tende a mantê-la.
* Ser copiada entre donos quase não importa (OR 0,97).

**Cuidado:** com 788 mil observações tudo é estatisticamente significativo, mas o modelo explica pouco (pseudo R² = 0,03). São **associações modestas**, não previsões. E a autoria por assistente só pode ser 1 onde há histórico (quase só em `.claude/skills/`), então parte do efeito pode ser o local.

## Conclusão da RQ5
* Detectamos alguma capacidade de gravidade alta em 6,4% das skills; descontada a precisão dos detectores, são **no máximo ~2%**. A capacidade confirmada mais comum é instalar pacotes (~10%).
* **17,3%** das variantes acrescentam alguma capacidade, mas as edições, no geral, **mais removem do que acrescentam**.
* A exceção confiável é **`curl | sh`**, que cresce nas cópias. A aparente alta da execução dinâmica é um artefato do detector ("eval" = avaliação).
* Variantes com scripts ou escritas com assistente acrescentam capacidades com mais frequência.

**Limitações:** (1) a precisão vem de 12 trechos por categoria, então os intervalos são largos (cerca de ±25 pontos percentuais); o avaliador 1 foi uma pré-rotulagem com apoio de um modelo de linguagem, e o κ nos trechos rotulados às cegas é moderado (0,54); (2) os detectores de segredos, execução dinâmica e desligar confirmações têm precisão baixa, e os números dessas categorias são reportados com essa ressalva; (3) famílias agrupadas pelo nome juntam skills sem relação; (4) a referência é um proxy da original e, em muitas famílias, tem só 1 cópia ou empata com outras.

**Próximo passo:** o notebook 05 mede quanto confiar nos detectores.